# Groupby and Pivot Tables

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox

from IPython.display import display, clear_output

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this session you will be able to:

- **Apply** `groupby()` with aggregation functions to summarize a numeric column across groups
- **Interpret** grouped statistics to identify which groups differ most
- **Construct** a pivot table that cross-tabulates two categorical variables
- **Distinguish** when to use `groupby()`, `pivot_table()`, and `crosstab()`

> **Tip:** Start with `pclass` as the group and `survived` as the metric. Notice the survival rate climbs sharply from 3rd to 1st class. Then switch to `sex` and watch the gap widen further. This is bivariate analysis done in two lines of pandas.

---
## How we got here

In `07_descriptive_statistics_in_practice.ipynb` we computed mean, median, and standard deviation for the whole dataset. That told us the average fare across all 891 passengers. But that single number mixes first-class tickets with steerage. `groupby()` lets us ask: what is the average fare for each class separately? From here, every bivariate and multivariate insight in the EDA curriculum flows from this one operation.

---
## Why this matters for data science

`groupby()` is probably the single most-used pandas operation in EDA. Before you train a model, you need to know whether your outcome variable behaves differently across groups. If the survival rate is 63% in first class and 24% in third class, class is almost certainly a useful feature. If the survival rate were 38% in every class, class would tell the model nothing.

Pivot tables extend this to two dimensions at once. They answer questions like: does the survival gap between men and women hold across all three passenger classes, or does it only appear in one of them? The answer shapes which features you build and which interactions you add to your model.

---
## Try it yourself

In [ ]:
out = Output()

group_dropdown = Dropdown(
    options=["pclass", "sex", "embarked"],
    value="pclass",
    description="Group by:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)

value_dropdown = Dropdown(
    options=["survived", "fare", "age", "sibsp", "parch"],
    value="survived",
    description="Metric:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px"),
)

agg_dropdown = Dropdown(
    options=["mean", "median", "count", "sum", "std"],
    value="mean",
    description="Aggregation:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="240px"),
)

def render(change=None):
    group = group_dropdown.value
    col   = value_dropdown.value
    func  = agg_dropdown.value

    result = titanic.groupby(group)[col].agg(func).reset_index()
    metric_col = f"{func}({col})"
    result.columns = [group, metric_col]

    fig = go.Figure(
        go.Bar(
            x=result[group].astype(str),
            y=result[metric_col],
            marker_color=PALETTE["primary"],
            text=result[metric_col].round(3),
            textposition="outside",
            width=0.5,
        )
    )
    layout = base_layout(
        title=f"{func.capitalize()} of {col} grouped by {group}",
        xaxis_title=group,
        yaxis_title=metric_col,
    )
    layout.update(showlegend=False)
    fig.update_layout(layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

group_dropdown.observe(render, names="value")
value_dropdown.observe(render, names="value")
agg_dropdown.observe(render, names="value")

display(VBox([HBox([group_dropdown, value_dropdown, agg_dropdown]), out]))
render()

---
## What's happening?

Every `groupby()` operation follows the same three steps:

1. **Split** — the dataset is divided into groups based on one column
2. **Apply** — a function is computed within each group independently
3. **Combine** — the results are assembled back into a new Series or DataFrame

```python
titanic.groupby("pclass")["survived"].mean()
```

- `groupby("pclass")` splits passengers into three groups
- `["survived"]` selects the column to analyze within each group
- `.mean()` computes the aggregation and combines the results

The choice of aggregation function changes the question you are asking:

| Function | Question it answers |
|----------|---------------------|
| `mean` | What is the average outcome per group? |
| `median` | What is the typical outcome, resistant to outliers? |
| `count` | How many rows are in each group? |
| `sum` | What is the total across each group? |
| `std` | How much does the outcome vary within each group? |

When `mean` and `median` diverge within a group, that group's distribution is skewed — the same rule from `07_descriptive_statistics_in_practice.ipynb` applies here, just within each subgroup.

---
## Real-world example: Pivot Tables

The chart below shows survival rate broken down by both passenger class and sex — a two-dimensional groupby, which pandas calls a pivot table.

Notice:
- Women in 1st class survived at a 97% rate
- Men in 3rd class survived at a 14% rate
- The sex gap holds across all three classes, not just one

This two-variable view is far more informative than either variable alone. It is the kind of question a single `groupby()` call cannot answer directly. You would need `groupby(["pclass", "sex"])`, which produces a hierarchical index. `pivot_table()` produces the same result in a cleaner, readable 2D format.

> **Discussion question:** Looking at this pivot table, which variable appears to be the stronger predictor of survival: class or sex? What evidence supports your answer?

In [ ]:
pivot = pd.pivot_table(
    titanic,
    values="survived",
    index="pclass",
    columns="sex",
    aggfunc="mean",
)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=[f"{p}-class" for p in pivot.index.tolist()],
    colorscale=[[0, PALETTE["secondary"]], [1, PALETTE["primary"]]],
    text=[[f"{v:.0%}" for v in row] for row in pivot.values],
    texttemplate="%{text}",
    showscale=True,
    colorbar=dict(title="Survival Rate"),
))
layout = base_layout(
    title="Survival Rate by Class and Sex (Pivot Table)",
    xaxis_title="Sex",
    yaxis_title="Passenger Class",
)
fig.update_layout(layout)
fig.show()

---
## Three tools for the same job

All three tools below summarize data by group. The difference is how many grouping dimensions you need and what you are aggregating.

| Tool | Best for | Example |
|------|----------|---------|
| `groupby().agg()` | One group column, any aggregation | Mean fare per class |
| `pivot_table()` | Two group columns, any aggregation | Survival rate by class × sex |
| `crosstab()` | Counting combinations of two categoricals | Passenger count per class × sex |

```python
# groupby — one group column
titanic.groupby("pclass")["fare"].mean()

# pivot_table — two group columns
pd.pivot_table(titanic, values="fare", index="pclass", columns="sex", aggfunc="mean")

# crosstab — count combinations of two categoricals
pd.crosstab(titanic["pclass"], titanic["survived"])
```

Use `groupby()` by default. Reach for `pivot_table()` when you need a 2D comparison. Use `crosstab()` when you only need counts.

---
## Watch out: Simpson's Paradox

A trend that appears in grouped data can reverse or disappear when you add a third variable. This is Simpson's Paradox, and `groupby()` is often where you first encounter it.

Here is a real example from the Titanic. Passengers who embarked at Cherbourg (C) appear to have the highest survival rate when you group by port alone. That seems to suggest something about Cherbourg, but the explanation is much simpler.

Cherbourg boarded a disproportionately high share of first-class passengers. First-class passengers survived at much higher rates than third-class passengers. The apparent Cherbourg advantage is entirely explained by class composition. Embarkation port itself has almost no direct relationship with survival.

The lesson: a pattern you find with `groupby()` on one variable is a candidate explanation, not a confirmed one. Always check whether a third variable is driving the result before drawing conclusions.

> **Discussion question:** If you only ran `titanic.groupby("embarked")["survived"].mean()` and stopped there, what conclusion might you draw? What does the pivot table below reveal that changes that conclusion?

In [ ]:
# Chart 1 — overall survival rate by embarkation port
port_labels = {"C": "Cherbourg (C)", "Q": "Queenstown (Q)", "S": "Southampton (S)"}

surv_by_port = (
    titanic.dropna(subset=["embarked"])
    .groupby("embarked")["survived"]
    .mean()
    .reset_index()
)
surv_by_port.columns = ["embarked", "survival_rate"]
surv_by_port["label"] = surv_by_port["embarked"].map(port_labels)

fig1 = go.Figure(
    go.Bar(
        x=surv_by_port["label"],
        y=surv_by_port["survival_rate"] * 100,
        marker_color=PALETTE["primary"],
        text=(surv_by_port["survival_rate"] * 100).round(1).astype(str) + "%",
        textposition="outside",
        width=0.5,
    )
)
layout1 = base_layout(
    title="Survival Rate by Embarkation Port",
    xaxis_title="Port",
    yaxis_title="Survival Rate (%)",
)
layout1.update(showlegend=False, yaxis=dict(range=[0, 65]))
fig1.update_layout(layout1)
fig1.show()

# Chart 2 — survival by port x class reveals the confound
pivot_sp = pd.pivot_table(
    titanic.dropna(subset=["embarked"]),
    values="survived",
    index="embarked",
    columns="pclass",
    aggfunc="mean",
)

fig2 = go.Figure(go.Heatmap(
    z=pivot_sp.values,
    x=[f"{c}-class" for c in pivot_sp.columns.tolist()],
    y=[port_labels.get(p, p) for p in pivot_sp.index.tolist()],
    colorscale=[[0, PALETTE["secondary"]], [1, PALETTE["primary"]]],
    text=[
        [f"{v:.0%}" if not pd.isna(v) else "n/a" for v in row]
        for row in pivot_sp.values
    ],
    texttemplate="%{text}",
    showscale=True,
    colorbar=dict(title="Survival Rate"),
))
layout2 = base_layout(
    title="Survival Rate by Port × Class (the confound revealed)",
    xaxis_title="Passenger Class",
    yaxis_title="Embarkation Port",
)
fig2.update_layout(layout2)
fig2.show()

---
## Key takeaway

> **`groupby()` and `pivot_table()` are the same operation: one groups on one axis, the other groups on two. Mastering both means you can answer any "how does X differ by group?" question in two lines of pandas and check whether that answer holds up when you add a third variable.**

---
*Next up: 08_univariate_analysis.ipynb — visualizing each variable one at a time to understand its full distribution*